# Relax structure with MACE — Machine Learned Interatomic Potential

Use Zur and McGill superlattices matching [algorithm](https://doi.org/10.1063/1.3330840) to create interfaces between two materials using the `mat3ra-made` [implementation](https://github.com/Exabyte-io/made).

<h2 style="color:green">Usage</h2>

1. Drop the materials files into the "uploads" folder in the JupyterLab file browser
1. Set Input Parameters below or use the default values
1. Click "Run" > "Run All" to run all cells
1. Wait for the run to complete. Scroll down to view cell results.
1. Review the strain plot and modify parameters as needed

## Methodology

1. Load materials from JSON files and create substrate and film slabs via `mat3ra-made`
2. Run ZSL strain matching and plot strain vs interface area
3. Create the selected interface and convert to ASE atoms with `to_ase()`
4. Relax the interface with MACE-MP-0 and visualize convergence
5. Compute interface binding energy decomposition

## 1. Set Input Parameters

### 1.1. Substrate and Film Materials

In [ ]:

SUBSTRATE_NAME = "Nickel"
FILM_NAME = "Graphene"
SUBSTRATE_INDEX = 0
FILM_INDEX = 1

### 1.2. Slab Parameters

In [ ]:
SUBSTRATE_MILLER_INDICES = (1, 1, 1)
SUBSTRATE_THICKNESS = 6  # in atomic layers
SUBSTRATE_TERMINATION_FORMULA = None  # if None, the first termination is used

FILM_MILLER_INDICES = (0, 0, 1)
FILM_THICKNESS = 1  # in atomic layers
FILM_TERMINATION_FORMULA = None  # if None, the first termination is used

USE_CONVENTIONAL_CELL = True

### 1.3. Interface and Relaxation Parameters

In [ ]:
INTERFACE_DISTANCE = None  # gap between substrate and film, in Angstrom, if None, the distance will be set to the sum of the covalent radii of the two materials
INTERFACE_VACUUM = 10.0  # vacuum over film, in Angstrom
REDUCE_RESULT_CELL_TO_PRIMITIVE = True

MAX_AREA = 150  # in Angstrom^2
MAX_AREA_TOLERANCE = 0.09
MAX_LENGTH_TOLERANCE = 0.05
MAX_ANGLE_TOLERANCE = 0.02

RELAXATION_PARAMETERS = {
    "FMAX": 0.05,
}
MACE_MODEL = "large"  # "small", "medium", or "large" MACE-MP-0 foundation model

## 2. Install Packages

In [ ]:

try:
    from pyodide.http import pyfetch as _pyodide_pyfetch
except ImportError:
    _pyodide_pyfetch = None

# Mirrors mace.calculators.foundations_models.mace_mp_urls (mace-torch 0.3.x).
MODEL_PATHS_MAP = {
#    "small": "https://github.com/ACEsuit/mace-mp/releases/download/mace_mp_0/2023-12-10-mace-128-L0_energy_epoch-249.model",
#     "small": "/drive/packages/models/2023-12-10-mace-128-L0_energy_epoch-249.model",
#    "medium": "https://github.com/ACEsuit/mace-mp/releases/download/mace_mp_0/2023-12-03-mace-128-L1_epoch-199.model",
    "large": "/drive/packages/models/MACE_MPtrj_2022.9.model",
#    "large": "https://github.com/ACEsuit/mace-mp/releases/download/mace_mp_0/MACE_MPtrj_2022.9.model",
#     "small-0b": "https://github.com/ACEsuit/mace-mp/releases/download/mace_mp_0b/mace_agnesi_small.model",
#     "medium-0b": "https://github.com/ACEsuit/mace-mp/releases/download/mace_mp_0b/mace_agnesi_medium.model",
#     "small-0b2": "https://github.com/ACEsuit/mace-mp/releases/download/mace_mp_0b2/mace-small-density-agnesi-stress.model",
#     "medium-0b2": "https://github.com/ACEsuit/mace-mp/releases/download/mace_mp_0b2/mace-medium-density-agnesi-stress.model",
#     "large-0b2": "https://github.com/ACEsuit/mace-mp/releases/download/mace_mp_0b2/mace-large-density-agnesi-stress.model",
#     "medium-0b3": "https://github.com/ACEsuit/mace-mp/releases/download/mace_mp_0b3/mace-mp-0b3-medium.model",
#     "medium-mpa-0": "https://github.com/ACEsuit/mace-mp/releases/download/mace_mpa_0/mace-mpa-0-medium.model",
#     "small-omat-0": "https://github.com/ACEsuit/mace-mp/releases/download/mace_omat_0/mace-omat-0-small.model",
#     "medium-omat-0": "https://github.com/ACEsuit/mace-mp/releases/download/mace_omat_0/mace-omat-0-medium.model",
#     "mace-matpes-pbe-0": "https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-pbe-omat-ft.model",
#     "mace-matpes-r2scan-0": "https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-r2scan-omat-ft.model",
}

def _matscipy_neighbour_list_compat(quantities, atoms=None, cutoff=None, positions=None, cell=None, pbc=None, **_):
    """Translate matscipy-style keyword-arg call into an ASE neighbor_list call."""
    from ase import Atoms as _Atoms
    from ase.neighborlist import neighbor_list as _ase_neighbor_list
    if atoms is None:
        atoms = _Atoms(positions=positions, cell=cell, pbc=pbc if pbc is not None else [False, False, False])
    return _ase_neighbor_list(quantities, atoms, cutoff)


class _LoggingTensorModeStub:
    def __enter__(self):
        return self

    def __exit__(self, *args):
        return False


def _capture_logs_stub(*args, **kwargs):
    return _LoggingTensorModeStub()


def patch_mace_for_pyodide():

    """Stub modules absent in pyodide's stripped torch build and missing C-extension packages."""
    _internal = types.ModuleType("torch.testing._internal")
    _internal.__path__ = []  # __path__ marks it as a package so sub-imports resolve
    _internal.__package__ = "torch.testing._internal"

    _common_utils = types.ModuleType("torch.testing._internal.common_utils")
    _common_utils.dtype_abbrs = {}

    _logging_tensor = types.ModuleType("torch.testing._internal.logging_tensor")
    _logging_tensor.LoggingTensorMode = _LoggingTensorModeStub
    _logging_tensor.capture_logs = _capture_logs_stub

    _internal.common_utils = _common_utils
    _internal.logging_tensor = _logging_tensor
    sys.modules["torch.testing._internal"] = _internal
    sys.modules["torch.testing._internal.common_utils"] = _common_utils
    sys.modules["torch.testing._internal.logging_tensor"] = _logging_tensor

    _matscipy = types.ModuleType("matscipy")
    _matscipy.__path__ = []
    _matscipy.__package__ = "matscipy"
    _matscipy_neighbours = types.ModuleType("matscipy.neighbours")
    _matscipy_neighbours.neighbour_list = _matscipy_neighbour_list_compat
    _matscipy.neighbours = _matscipy_neighbours
    sys.modules["matscipy"] = _matscipy
    sys.modules["matscipy.neighbours"] = _matscipy_neighbours
    
    
    # lmdb, orjson, h5py are C-extension packages that cannot be compiled in
    # Emscripten (no subprocess / cc support).  They are only used by the
    # training/dataset-loading paths (mace.data.lmdb_dataset,
    # mace.data.hdf5_dataset) which are pulled in by mace.data.__init__ but
    # are never exercised during Pyodide inference.  Stubs let the import
    # succeed; any actual call to these APIs would raise at runtime, which is
    # the correct behaviour for an unsupported operation.
    for _pkg in ("lmdb", "h5py"):
        if _pkg not in sys.modules:
            sys.modules[_pkg] = types.ModuleType(_pkg)

    # mace.data.atomic_data accesses torch_geometric.data.Data at class-definition
    # time.  In Pyodide the submodule attribute can be unset when mace.tools.__init__
    # is still mid-import (circular via mace.cli.visualise_train) at the point where
    # train.py runs `from . import torch_geometric` – so the subpackage exists in
    # sys.modules but .data has not yet been stamped onto it.  Pre-importing here,
    # with the matscipy stub already in place, forces the full init to complete and
    # guarantees the attribute is set before any mace.data import runs.
    try:
        import importlib as _importlib
        _tg = _importlib.import_module("mace.tools.torch_geometric")
        _tg_data = _importlib.import_module("mace.tools.torch_geometric.data")
        _tg.data = _tg_data
    except Exception:
        pass



async def download_mace_model_pyodide(model = None) -> str:
    """Download a MACE foundation model in Pyodide using the browser's fetch API.

    urllib.request does not support HTTPS in Pyodide WASM; this function uses
    pyodide.http.pyfetch instead, which delegates to the browser's native fetch.

    Args:
        model: MACE model name (e.g. "medium", "medium-mpa-0") or a direct HTTPS URL.
                None defaults to "medium-mpa-0".
    Returns:
        Local filesystem path to the downloaded model file.
    """
    if _pyodide_pyfetch is None:
        raise RuntimeError("pyodide.http.pyfetch unavailable; call mace_mp() directly in standard Python.")
    url = MODEL_PATHS_MAP.get(model or "medium-mpa-0", model)
    if url is None or not url.startswith("http"):
        return model  # already a local path
    cache_dir = os.path.join(os.path.expanduser("~"), ".cache", "mace")
    os.makedirs(cache_dir, exist_ok=True)
    filename = "".join(c for c in os.path.basename(url) if c.isalnum() or c in "_")
    local_path = os.path.join(cache_dir, filename)
    if not os.path.isfile(local_path):
        print(f"Downloading MACE model from {url!r}")
        response = await _pyodide_pyfetch(url)
        with open(local_path, "wb") as f:
            f.write(await response.bytes())
        print(f"Cached MACE model to {local_path}")
    return local_path



In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip


    await micropip.install("mat3ra-api-examples", deps=False)
    await micropip.install("mat3ra-utils")
    from mat3ra.utils.jupyterlite.packages import install_packages
  
    import types

    await install_packages("api_examples|torch")
    await micropip.install("torch-dftd")

    await micropip.install("opt_einsum")
    await micropip.install("opt_einsum_fx", deps=False)
    await micropip.install("e3nn==0.4.4", deps=False)

    # MACE training deps
    await micropip.install("prettytable")
    await micropip.install("torch_ema", deps=False)
    await micropip.install("lightning-utilities", deps=False)
    await micropip.install("torchmetrics", deps=False)
    
    #
    await micropip.install("ssl")
    await micropip.install("h5py")
    await micropip.install("lmdb")

    await micropip.install("orjson")
    await micropip.install("anywidget")
    
    # MACE
    await micropip.install("mace-torch", deps=False)
    from utils import patch_torch


## 3. Load Materials

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials

substrate = Material.create(Materials.get_by_name_first_match(SUBSTRATE_NAME))
film = Material.create(Materials.get_by_name_first_match(FILM_NAME))

print("Substrate:", substrate.name)
print("Film:     ", film.name)

### 3.1. Visualize Input Materials

In [ ]:
from utils.visualize import visualize_materials as visualize

visualize([substrate, film], repetitions=[3, 3, 1], rotation="0x")

## 3.2. Calculate nearest neighbor distance for each material to inform interface distance choice


In [ ]:
from mat3ra.made.tools.build_components.entities.reusable.three_dimensional.supercell.helpers import create_supercell
from mat3ra.made.tools.analyze.rdf import RadialDistributionFunction
### 7.3. Plot Radial Distribution Functions
from utils.plot import plot_rdf

substrate_supercell = create_supercell(substrate, scaling_factor=[3, 3, 3])
film_supercell = create_supercell(film, scaling_factor=[3, 3, 3])

rdf_substrate = RadialDistributionFunction.from_material(substrate_supercell, cutoff=5.0)
rdf_film = RadialDistributionFunction.from_material(film_supercell, cutoff=5.0)

first_peak_substrate = rdf_substrate.first_peak_distance
first_peak_film = rdf_film.first_peak_distance

print(f"First RDF peak for substrate ({substrate.name}): {first_peak_substrate:.3f} Å")
print(f"First RDF peak for film ({film.name}): {first_peak_film:.3f} Å")

if INTERFACE_DISTANCE is None:
    INTERFACE_DISTANCE = (first_peak_substrate + first_peak_film) / 2
    print(f"Setting interface distance to {INTERFACE_DISTANCE:.3f} Å based on RDF peaks")

# plot_rdf(substrate_supercell, cutoff=5.0)
# plot_rdf(film_supercell, cutoff=5.0)

## 4. Configure Slabs

### 4.1. Get Possible Terminations

In [ ]:
from mat3ra.made.tools.helpers import get_slab_terminations

film_slab_terminations = get_slab_terminations(material=film, miller_indices=FILM_MILLER_INDICES)
substrate_slab_terminations = get_slab_terminations(material=substrate, miller_indices=SUBSTRATE_MILLER_INDICES)
print("Film slab terminations:     ", film_slab_terminations)
print("Substrate slab terminations:", substrate_slab_terminations)

### 4.2. Visualize Slabs for All Possible Terminations

In [ ]:
from mat3ra.made.tools.helpers import create_slab, select_slab_termination
from mat3ra.made.tools.helpers import create_interface_zsl_between_slabs

film_slabs = [
    create_slab(film, miller_indices=FILM_MILLER_INDICES, termination_top=t, vacuum=0)
    for t in film_slab_terminations
]
substrate_slabs = [
    create_slab(substrate, miller_indices=SUBSTRATE_MILLER_INDICES, termination_top=t, vacuum=0, number_of_layers=4)
    for t in substrate_slab_terminations
]

# visualize(
#     [{"material": s, "title": str(t)} for s, t in zip(film_slabs, film_slab_terminations)],
#     repetitions=[3, 3, 1], rotation="-90x",
# )
# visualize(
#     [{"material": s, "title": str(t)} for s, t in zip(substrate_slabs, substrate_slab_terminations)],
#     repetitions=[3, 3, 1], rotation="-90x",
# )

### 4.3. Create Substrate and Film Slabs

In [ ]:
from mat3ra.made.tools.build.pristine_structures.two_dimensional.slab import SlabConfiguration, SlabBuilder

substrate_slab_config = SlabConfiguration.from_parameters(
    material_or_dict=substrate,
    miller_indices=SUBSTRATE_MILLER_INDICES,
    number_of_layers=SUBSTRATE_THICKNESS,
    vacuum=0.0,
    termination_top_formula=SUBSTRATE_TERMINATION_FORMULA,
    use_conventional_cell=USE_CONVENTIONAL_CELL,
)
film_slab_config = SlabConfiguration.from_parameters(
    material_or_dict=film,
    miller_indices=FILM_MILLER_INDICES,
    number_of_layers=FILM_THICKNESS,
    vacuum=0.0,
    termination_bottom_formula=FILM_TERMINATION_FORMULA,
    use_conventional_cell=USE_CONVENTIONAL_CELL,
)

substrate_slab = SlabBuilder().get_material(substrate_slab_config)
film_slab = SlabBuilder().get_material(film_slab_config)

## 5. Find Interfaces with ZSL Strain Matching

### 5.1. Initialize ZSL Analyzer

In [ ]:
from mat3ra.made.tools.analyze.interface import ZSLInterfaceAnalyzer

zsl_analyzer = ZSLInterfaceAnalyzer(
    substrate_slab_configuration=substrate_slab_config,
    film_slab_configuration=film_slab_config,
    max_area=MAX_AREA,
    max_area_ratio_tol=MAX_AREA_TOLERANCE,
    max_length_tol=MAX_LENGTH_TOLERANCE,
    max_angle_tol=MAX_ANGLE_TOLERANCE,
    reduce_result_cell=False,
)

### 5.2. Generate and Plot Matches

In [ ]:
from utils.plot import plot_strain_vs_area

PLOT_SETTINGS = {
    "HEIGHT": 600,
    "X_SCALE": "log",
    "Y_SCALE": "log",
}

matches = zsl_analyzer.zsl_match_holders
print(f"Found {len(matches)} matches")
# plot_strain_vs_area(matches, PLOT_SETTINGS)

### 5.3. Select the Interface

Choose the match index from the plot above (index 0 has the lowest strain).

In [ ]:
selected_index = 0

## 6. Create the Interface

In [ ]:
interface = create_interface_zsl_between_slabs(
    substrate_slab=substrate_slab,
    film_slab=film_slab,
    gap=INTERFACE_DISTANCE,
    vacuum=INTERFACE_VACUUM,
    match_id=selected_index,
    max_area=MAX_AREA,
    max_area_ratio_tol=MAX_AREA_TOLERANCE,
    max_length_tol=MAX_LENGTH_TOLERANCE,
    max_angle_tol=MAX_ANGLE_TOLERANCE,
    reduce_result_cell_to_primitive=REDUCE_RESULT_CELL_TO_PRIMITIVE,
)

### 6.1. Visualize Interface

In [ ]:
from utils.visualize import ViewersEnum

# visualize([{"material": interface, "title": interface.name}], viewer=ViewersEnum.wave)
# visualize(interface, repetitions=[1, 1, 1], rotation="-90x")

## 7. Apply Relaxation
### 7.1. Relax with MACE

In [ ]:

import os
import plotly.graph_objs as go
from IPython.display import display
from plotly.subplots import make_subplots

from mat3ra.made.tools.convert import to_ase
from ase.optimize import BFGS

from mace.calculators import MACECalculator

calculator = MACECalculator(model_path=MODEL_PATHS_MAP["large"], dispersion=False, default_dtype="float32", device="cpu")

ase_interface = to_ase(interface)
ase_interface.set_calculator(calculator)
dyn = BFGS(ase_interface)

steps = []
energies = []

fig = make_subplots(rows=1, cols=1, specs=[[{"type": "scatter"}]])
scatter = go.Scatter(x=[], y=[], mode="lines+markers", name="Energy")
fig.add_trace(scatter)
fig.update_layout(title_text="Real-time Optimization Progress", xaxis_title="Step", yaxis_title="Energy (eV)")

f = go.FigureWidget(fig)
display(f)


def plotly_callback():
    step = dyn.nsteps
    energy = ase_interface.get_total_energy()
    steps.append(step)
    energies.append(energy)
    print(f"Step: {step}, Energy: {energy:.4f} eV")
    with f.batch_update():
        f.data[0].x = steps
        f.data[0].y = energies


dyn.attach(plotly_callback, interval=1)
dyn.run(fmax=RELAXATION_PARAMETERS["FMAX"])

ase_original_interface = to_ase(interface)
ase_original_interface.set_calculator(calculator)
ase_final_interface = ase_interface

original_energy = ase_original_interface.get_total_energy()
relaxed_energy = ase_interface.get_total_energy()

# print("Original structure:\n", ase_to_poscar(ase_original_interface))
# print("\nRelaxed structure:\n", ase_to_poscar(ase_final_interface))
print(f"The final energy is {float(relaxed_energy):.3f} eV.")

### 7.2. View Structure Before and After Relaxation

In [ ]:
from mat3ra.made.tools.convert import from_ase


def atoms_to_material(atoms, title):
    material = Material.create(from_ase(atoms))
    material.name = title
    return material


material_original = atoms_to_material(ase_original_interface, f"Original  E={original_energy:.3f} eV")
material_relaxed = atoms_to_material(ase_final_interface, f"Relaxed  E={relaxed_energy:.3f} eV")

visualize(
    [
        {"material": material_original, "title": material_original.name},
        {"material": material_relaxed, "title": material_relaxed.name},
    ],
    viewer=ViewersEnum.wave,
)

## 7.4. Output interlayer distance before and after relaxation

In [ ]:
from mat3ra.made.tools.analyze.other import get_average_interlayer_distance

print(f"Interlayer distance before relaxation: {get_average_interlayer_distance(material_original, 0, 1):.4f} Å")
print(f"Interlayer distance after relaxation:  {get_average_interlayer_distance(material_relaxed, 0, 1):.4f} Å")

### 7.4. Calculate Interface Energy

In [ ]:
def filter_atoms_by_tag(atoms, material_index):
    return atoms[atoms.get_tags() == material_index]


def calculate_energy(atoms, calc):
    atoms.set_calculator(calc)
    return atoms.get_total_energy()


def calculate_delta_energy(total_energy, *component_energies):
    return total_energy - sum(component_energies)


substrate_original = filter_atoms_by_tag(ase_original_interface, SUBSTRATE_INDEX)
layer_original = filter_atoms_by_tag(ase_original_interface, FILM_INDEX)
substrate_relaxed = filter_atoms_by_tag(ase_final_interface, SUBSTRATE_INDEX)
layer_relaxed = filter_atoms_by_tag(ase_final_interface, FILM_INDEX)

original_substrate_energy = calculate_energy(substrate_original, calculator)
original_layer_energy = calculate_energy(layer_original, calculator)
relaxed_substrate_energy = calculate_energy(substrate_relaxed, calculator)
relaxed_layer_energy = calculate_energy(layer_relaxed, calculator)

delta_original = calculate_delta_energy(original_energy, original_substrate_energy, original_layer_energy)
delta_relaxed = calculate_delta_energy(relaxed_energy, relaxed_substrate_energy, relaxed_layer_energy)

area = ase_original_interface.get_volume() / ase_original_interface.cell[2, 2]
n_interface = ase_final_interface.get_global_number_of_atoms()
n_substrate = substrate_relaxed.get_global_number_of_atoms()
n_layer = layer_relaxed.get_global_number_of_atoms()
effective_delta_relaxed = (
                                  relaxed_energy / n_interface
                                  - (relaxed_substrate_energy / n_substrate + relaxed_layer_energy / n_layer)
                          ) / (2 * area)

print(f"Original Substrate energy: {original_substrate_energy:.4f} eV")
print(f"Relaxed Substrate energy:  {relaxed_substrate_energy:.4f} eV")
print(f"Original Layer energy:     {original_layer_energy:.4f} eV")
print(f"Relaxed Layer energy:      {relaxed_layer_energy:.4f} eV")
print("\nDelta between interface energy and sum of component energies")
print(f"Original Delta:            {delta_original:.4f} eV")
print(f"Relaxed Delta:             {delta_relaxed:.4f} eV")
print(f"Original Delta per area:   {delta_original / area:.4f} eV/Ang^2")
print(f"Relaxed Delta per area:    {delta_relaxed / area:.4f} eV/Ang^2")
print(f"Relaxed interface energy:  {relaxed_energy:.4f} eV")
print(
    f"Effective relaxed Delta per area: {effective_delta_relaxed:.4f} eV/Ang^2 ({effective_delta_relaxed / 0.16:.4f} J/m^2)")

## References

[1] mat3ra-made interface builder: https://github.com/Exabyte-io/made  
[2] MACE-MP-0 foundation model: https://github.com/ACEsuit/mace?tab=readme-ov-file#foundation-models  